Run gold transformations and checks in the default lakehouse.


In [ ]:
import os
import shutil
import subprocess
import sys

import notebookutils

shutil.copytree("/lakehouse/default/Files/code", "/tmp/code", dirs_exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

context = notebookutils.runtime.context
env = os.environ.copy()
env.update(
    LAKE_ROOT="/lakehouse/default",
    LAKE_WRITE_URL=(
        f"abfss://{context['currentWorkspaceId']}@onelake.dfs.fabric.microsoft.com/"
        f"{context['defaultLakehouseId']}"
    ),
    AZURE_STORAGE_TOKEN=notebookutils.credentials.getToken("storage"),
)

for module in [("pipeline.transform", "gold"), ("pipeline.checks", "gold")]:
    subprocess.run(
        ["uv", "run", "--frozen", "--no-dev", "python", "-m", *module],
        cwd="/tmp/code", env=env, check=True,
    )
